In [1]:
"""
GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10
=============================================================
GPU-native PTQ using multiple frameworks, since PyTorch's static PTQ
is CPU-only. This script benchmarks three GPU-capable approaches:

  1. ONNX Runtime  — export FP32 → ONNX → INT8 via ORT quantization API
                     (CUDAExecutionProvider for inference)
  2. TensorRT      — export FP32 → ONNX → TRT INT8 engine with calibration
                     (requires tensorrt + pycuda; best GPU throughput)
  3. Torch CUDA    — dynamic PTQ (Linear → qint8) run on CUDA tensors;
                     PT2E (PyTorch 2 Export) + X86InductorQuantizer → torch.compile

FLOPs measurement:
  - fvcore.nn.FlopCountAnalysis for FP32/quantized PyTorch models
  - Manual formula for conv/linear: FLOPs = 2 × Cin × Cout × Kh × Kw × H × W
  - IMPORTANT: FLOPs (operation count) are IDENTICAL for FP32 and INT8 models.
    Quantization does not reduce the number of operations — it changes the
    datatype. The hardware throughput advantage of INT8 is ~4× on Ampere Tensor
    Cores because each SIMD lane processes 4× more INT8 elements per cycle.
    FLOPs and throughput are different quantities and must not be conflated.

Mathematical foundation (same as CPU PTQ, different execution backend):
  Q(x)  = round(x / S) + Z              [quantization]
  x̂     = S · (Q(x) − Z)               [dequantization]
  ε     = x − x̂,  |ε| ≤ S/2           [bounded error]
  S     = (x_max − x_min) / (2^b − 1)  [scale — from calibration data]
  Z     = round(−x_min / S)             [zero-point]

GPU throughput advantage (hardware parallelism, not fewer operations):
  - INT8 Tensor Core GEMM: ~4× throughput vs FP32 on Ampere (A100, RTX 30xx)
  - INT8 Tensor Core GEMM: ~8× throughput vs FP32 on Hopper (H100)
  - Memory bandwidth: ~4× reduction (1 byte INT8 vs 4 bytes FP32 per weight)
  - TensorRT layer fusion: Conv+BN+ReLU → single kernel, fewer memory round-trips
  - ORT CUDAExecutionProvider: uses cuDNN (Conv), cuBLASLt (MatMul), and custom
    kernels depending on op type and graph structure — not all ops use cuDNN INT8.

Output: gpu_ptq_results.json

Evaluation metrics recorded per backend (for study & comparison):
  accuracy          — top-1 classification accuracy
  precision         — macro-averaged precision (sklearn)
  recall            — macro-averaged recall   (sklearn)
  f1                — macro-averaged F1-score (sklearn)
  params_total      — total parameter count (integers)
  params_M          — parameter count in millions
  size_disk_mb      — serialised model size on disk (MB)
  inference_latency — dict: mean_ms, std_ms, min_ms, max_ms, throughput_imgs_per_sec
  flops_total       — total FLOPs (= 2 × MACs); identical for INT8 and FP32
  gflops            — flops_total / 1e9
"""

# ── Imports ────────────────────────────────────────────────────────────────────
import os
import sys
import json
import time
import copy
import math
import tempfile
import warnings
import argparse
import subprocess
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np

warnings.filterwarnings("ignore")

# ── Config ─────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "gpu_ptq_results_v4.json"

BATCH_SIZE     = 128
NUM_WORKERS    = 4
NUM_CLASSES    = 10
CALIB_SIZE     = 1024
CALIB_BATCHES  = 8
INFERENCE_RUNS = 50
INPUT_SHAPE    = (1, 3, 32, 32)   # single-image shape for export/FLOPs

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# ── Dependency detection ───────────────────────────────────────────────────────
def _try_import(name: str):
    try:
        import importlib
        return importlib.import_module(name)
    except ImportError:
        return None

torch         = _try_import("torch")
torchvision   = _try_import("torchvision")
ort           = _try_import("onnxruntime")
onnx          = _try_import("onnx")
onnxquant     = _try_import("onnxruntime.quantization")
fvcore        = _try_import("fvcore")
tensorrt      = _try_import("tensorrt")
pycuda        = _try_import("pycuda")
bitsandbytes  = _try_import("bitsandbytes")

# ── Install helpers (run once if needed) ───────────────────────────────────────
INSTALL_COMMANDS = {
    "torch":          "pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121",
    "onnx":           "pip install onnx onnxruntime-gpu",
    "onnxruntime":    "pip install onnxruntime-gpu",  # GPU build
    "fvcore":         "pip install fvcore",
    "tensorrt":       "pip install tensorrt",         # NVIDIA wheel
    "bitsandbytes":   "pip install bitsandbytes",
    "optimum":        "pip install optimum[onnxruntime-gpu]",
}

def install_deps():
    """Install all GPU PTQ dependencies. Run once before the experiment."""
    cmds = [
        "pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121",
        "pip install onnx onnxruntime-gpu",
        "pip install fvcore",
        "pip install tensorrt pycuda",
        "pip install bitsandbytes",
        "pip install optimum[onnxruntime-gpu]",
    ]
    for cmd in cmds:
        print(f"  $ {cmd}")
        subprocess.run(cmd.split(), check=False)


# ══════════════════════════════════════════════════════════════════════════════
# Model & Data
# ══════════════════════════════════════════════════════════════════════════════
def build_model(num_classes: int = 10):
    """ResNet-50 adapted for CIFAR-10 (32×32 input)."""
    import torch.nn as nn
    from torchvision import models
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path: str):
    model = build_model(NUM_CLASSES)
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    model.eval()
    return model

def get_test_loader():
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = datasets.CIFAR10(root="../data", train=False, download=True, transform=tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

def get_calib_loader():
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader, Subset
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds  = datasets.CIFAR10(root="../data", train=True, download=True, transform=tf)
    sub = Subset(ds, list(range(CALIB_SIZE)))
    return DataLoader(sub, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ══════════════════════════════════════════════════════════════════════════════
# Parameter Counting
# ══════════════════════════════════════════════════════════════════════════════
def count_parameters(model) -> Dict:
    """
    Count total and trainable parameters.
    Works for PyTorch nn.Module objects.
    Returns params_total (int) and params_M (float, millions).
    """
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "params_total"     : int(total),
        "params_trainable" : int(trainable),
        "params_M"         : round(total / 1e6, 3),
    }


# ══════════════════════════════════════════════════════════════════════════════
# FLOPs Measurement
# ══════════════════════════════════════════════════════════════════════════════
def count_flops_fvcore(model, input_shape=INPUT_SHAPE) -> Dict:
    """
    Use fvcore for accurate FLOPs counting.
    fvcore counts multiply-accumulate ops (MACs); we report GFLOPs = 2×MACs / 1e9.

    For a Conv2d layer:
      MACs = Cout × Cin × Kh × Kw × Hout × Wout
      FLOPs = 2 × MACs  (one multiply + one accumulate per MAC)

    For a Linear layer:
      MACs = Cout × Cin
      FLOPs = 2 × MACs
    """
    from fvcore.nn import FlopCountAnalysis, parameter_count
    x = torch.randn(*input_shape)
    flops = FlopCountAnalysis(model, x)
    flops.unsupported_ops_warnings(False)
    total_macs = flops.total()
    params     = parameter_count(model)[""]
    return {
        "total_macs"   : int(total_macs),
        "total_flops"  : int(total_macs * 2),
        "gflops_fp32"  : round(total_macs * 2 / 1e9, 4),
        "params_total" : int(params),
        "params_M"     : round(params / 1e6, 3),
        "hardware_throughput_context": {
            "flops_unchanged_by_quantization": True,
            "int8_tensor_core_speedup_ampere": "~4× throughput vs FP32 (hardware parallelism)",
            "int8_tensor_core_speedup_hopper": "~8× throughput vs FP32 (hardware parallelism)",
            "memory_bandwidth_reduction"     : "~4× (1 byte INT8 vs 4 bytes FP32 per weight)",
            "correct_framing": (
                "Quantization reduces memory bandwidth and enables wider SIMD. "
                "It does NOT reduce FLOPs. Report wall-clock latency for real speedup."
            ),
        },
        "note": (
            "FLOPs = 2 × MACs (one multiply + one accumulate per MAC). "
            "FLOPs are IDENTICAL for INT8 and FP32 models — quantization does not "
            "change the op count. Use measured latency to assess actual GPU speedup."
        ),
    }

def count_flops_manual(model) -> Dict:
    """
    Manual FLOPs calculation by walking named modules.
    Formula per layer:
      Conv2d  : FLOPs = 2 × Cin × Cout × Kh × Kw × Hout × Wout × N
      Linear  : FLOPs = 2 × in_features × out_features × N
    """
    import torch.nn as nn

    hooks, spatial = [], {}

    def make_hook(name):
        def hook(module, inp, out):
            if hasattr(out, "shape"):
                spatial[name] = tuple(out.shape)
        return hook

    handles = []
    for name, mod in model.named_modules():
        handles.append(mod.register_forward_hook(make_hook(name)))

    x = torch.randn(*INPUT_SHAPE)
    with torch.no_grad():
        model(x)
    for h in handles:
        h.remove()

    total_flops = 0
    layer_breakdown = {}

    for name, mod in model.named_modules():
        if isinstance(mod, nn.Conv2d):
            shape = spatial.get(name)
            if shape is None:
                continue
            _, cout, hout, wout = shape
            cin  = mod.in_channels
            kh   = mod.kernel_size[0] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            kw   = mod.kernel_size[1] if isinstance(mod.kernel_size, tuple) else mod.kernel_size
            macs = cout * (cin // mod.groups) * kh * kw * hout * wout
            flops = 2 * macs
            total_flops += flops
            layer_breakdown[name] = {"type": "Conv2d", "flops": flops, "macs": macs,
                                      "shape_out": list(shape)}

        elif isinstance(mod, nn.Linear):
            flops = 2 * mod.in_features * mod.out_features
            total_flops += flops
            layer_breakdown[name] = {"type": "Linear", "flops": flops,
                                      "in": mod.in_features, "out": mod.out_features}

    # Count parameters from the model directly
    params_total = sum(p.numel() for p in model.parameters())

    return {
        "total_flops"    : total_flops,
        "gflops"         : round(total_flops / 1e9, 4),
        "gflops_fp32"    : round(total_flops / 1e9, 4),  # alias for consistency
        "params_total"   : int(params_total),
        "params_M"       : round(params_total / 1e6, 3),
        "layer_breakdown": layer_breakdown,
        "formula": {
            "conv2d" : "FLOPs = 2 × (Cin/groups) × Cout × Kh × Kw × Hout × Wout",
            "linear" : "FLOPs = 2 × in_features × out_features",
            "batchnorm": "excluded — negligible vs Conv, fused by TRT/ORT, omitted by fvcore/thop",
        },
        "note": (
            "FLOPs are IDENTICAL for INT8 and FP32 models — quantization does not "
            "change the op count. INT8 throughput advantage is a hardware property "
            "(wider SIMD lanes), not a reduction in arithmetic operations."
        ),
    }

def measure_gpu_throughput_ms(model, device, n: int = INFERENCE_RUNS) -> Dict:
    """
    GPU latency using CUDA events (more accurate than time.perf_counter).
    Returns mean, std, min, max in milliseconds plus throughput in imgs/sec.
    """
    model = model.to(device).eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32, device=device)

    # Warmup
    with torch.no_grad():
        for _ in range(10):
            model(dummy)
    torch.cuda.synchronize(device)

    timings = []
    start_event = torch.cuda.Event(enable_timing=True)
    end_event   = torch.cuda.Event(enable_timing=True)

    with torch.no_grad():
        for _ in range(n):
            start_event.record()
            model(dummy)
            end_event.record()
            torch.cuda.synchronize()
            timings.append(start_event.elapsed_time(end_event))

    arr = np.array(timings)
    return {
        "mean_ms"               : round(float(arr.mean()), 4),
        "std_ms"                : round(float(arr.std()),  4),
        "min_ms"                : round(float(arr.min()),  4),
        "max_ms"                : round(float(arr.max()),  4),
        "samples"               : n,
        "batch_size"            : BATCH_SIZE,
        "throughput_imgs_per_sec": round(BATCH_SIZE / (float(arr.mean()) / 1000), 1),
        "timing_method"         : "CUDA events (GPU-synchronized)",
    }

def measure_cpu_ms(model, n: int = INFERENCE_RUNS) -> float:
    model = model.cpu().eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32)
    with torch.no_grad():
        for _ in range(5):
            model(dummy)
        t0 = time.perf_counter()
        for _ in range(n):
            model(dummy)
    return round((time.perf_counter() - t0) / n * 1000, 4)

def disk_size_mb(obj) -> float:
    """Works for PyTorch models and ONNX files (pass path as str)."""
    if isinstance(obj, str):
        return os.path.getsize(obj) / 1024 ** 2
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(obj, tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)


# ══════════════════════════════════════════════════════════════════════════════
# Shared helper: build the standard 8-metric result block
# ══════════════════════════════════════════════════════════════════════════════
def build_metrics_block(
    metrics: Dict,
    baseline_acc: float,
    model_or_path,          # nn.Module or str path — for disk size
    inference_latency: Dict,
    flops_fp32: Dict,
    params_override: Optional[Dict] = None,  # pass if model params unavailable
) -> Dict:
    """
    Build a standardised block containing all 8 evaluation metrics.
    This is included in every backend's result entry so every row has
    identical keys — making downstream comparison trivial.

    Fields
    ------
    accuracy          : top-1 accuracy  (float)
    precision         : macro precision (float)
    recall            : macro recall    (float)
    f1                : macro F1        (float)
    params_total      : int
    params_M          : float (millions)
    size_disk_mb      : float
    inference_latency : dict (mean_ms, std_ms, min_ms, max_ms,
                              throughput_imgs_per_sec, timing_method)
    flops_total       : int   — identical for INT8 and FP32
    gflops            : float — flops_total / 1e9
    accuracy_drop     : float — baseline_acc − accuracy
    compression_ratio : float — baseline_disk / size_disk_mb
    """
    disk_mb = disk_size_mb(model_or_path) if model_or_path is not None else None

    # Parameter count
    if params_override is not None:
        params_total = params_override.get("params_total", flops_fp32.get("params_total"))
        params_M     = params_override.get("params_M",     flops_fp32.get("params_M"))
    else:
        params_total = flops_fp32.get("params_total")
        params_M     = flops_fp32.get("params_M")

    # FLOPs — normalise key names across fvcore / manual / profiler outputs
    flops_total = (flops_fp32.get("total_flops")
                   or flops_fp32.get("flops_total"))
    gflops      = (flops_fp32.get("gflops_fp32")
                   or flops_fp32.get("gflops")
                   or (flops_total / 1e9 if flops_total else None))

    return {
        # ── Classification metrics ─────────────────────────────────────────
        "accuracy"         : round(metrics["accuracy"],  6),
        "precision"        : round(metrics["precision"], 6),
        "recall"           : round(metrics["recall"],    6),
        "f1"               : round(metrics["f1"],        6),
        "accuracy_drop"    : round(baseline_acc - metrics["accuracy"], 6),
        # ── Model size & parameters ────────────────────────────────────────
        "params_total"     : params_total,
        "params_M"         : params_M,
        "size_disk_mb"     : round(disk_mb, 4) if disk_mb is not None else None,
        # ── Inference time ─────────────────────────────────────────────────
        "inference_latency": inference_latency,   # full dict (mean, std, min, max, imgs/sec)
        # ── Operations ────────────────────────────────────────────────────
        "flops_total"      : flops_total,         # identical for INT8 and FP32
        "gflops"           : round(gflops, 4) if gflops else None,
        "flops_note"       : (
            "FLOPs are IDENTICAL for INT8 and FP32 — quantization changes the "
            "storage dtype and enables wider SIMD execution but does NOT reduce "
            "the arithmetic operation count. Use inference_latency.mean_ms "
            "for real-world GPU speedup comparison."
        ),
    }


# ══════════════════════════════════════════════════════════════════════════════
# Evaluation
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def evaluate_torch(model, loader, device) -> Dict:
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    model = model.to(device).eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(device)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def evaluate_onnx(session, loader) -> Dict:
    """Run inference with ONNX Runtime session."""
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    input_name = session.get_inputs()[0].name
    preds, labels = [], []
    for inputs, lbls in loader:
        out = session.run(None, {input_name: inputs.numpy()})[0]
        preds.extend(np.argmax(out, axis=1))
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


# ══════════════════════════════════════════════════════════════════════════════
# ONNX Export
# ══════════════════════════════════════════════════════════════════════════════
def export_to_onnx(model, output_path: str, opset: int = 17) -> str:
    """
    Export PyTorch FP32 model to ONNX.
    opset 17+ is required for INT8 quantization ops (QLinearConv, QLinearMatMul).
    Dynamic axes allow variable batch size at inference time.
    """
    dummy = torch.randn(*INPUT_SHAPE)
    torch.onnx.export(
        model.cpu().eval(),
        dummy,
        output_path,
        opset_version        = opset,
        input_names          = ["input"],
        output_names         = ["output"],
        dynamic_axes         = {"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        do_constant_folding  = True,
        export_params        = True,
    )
    print(f"        ✓ ONNX exported → {output_path}  "
          f"({os.path.getsize(output_path)/1024**2:.2f} MB, opset={opset})")
    return output_path


# ══════════════════════════════════════════════════════════════════════════════
# 1. ONNX Runtime — INT8 Static Quantization (GPU inference)
# ══════════════════════════════════════════════════════════════════════════════
def run_onnx_ptq(fp32_model, test_loader, calib_loader,
                 baseline_acc: float, base_disk: float,
                 device: str, flops_fp32: Dict) -> List[Dict]:
    import onnxruntime
    from onnxruntime.quantization import (
        quantize_static, CalibrationDataReader,
        QuantType, QuantFormat, CalibrationMethod
    )

    print("\n  [1/3] ONNX Runtime — Static INT8 PTQ")
    print("        Framework: onnxruntime-gpu (CUDAExecutionProvider)")
    print("        Kernels:   cuDNN INT8 (Conv), cuBLASLt (MatMul), custom (Add)")
    print("        S/Z:       calibrated once from calib data; frozen in graph")

    fp32_onnx = "resnet50_fp32.onnx"
    export_to_onnx(fp32_model, fp32_onnx)

    class CIFARCalibReader(CalibrationDataReader):
        def __init__(self, loader, max_batches=CALIB_BATCHES):
            self.data    = iter(loader)
            self.batches = 0
            self.max     = max_batches

        def get_next(self):
            if self.batches >= self.max:
                return None
            try:
                imgs, _ = next(self.data)
                self.batches += 1
                arr = np.ascontiguousarray(imgs.numpy(), dtype=np.float32)
                return {"input": arr}
            except StopIteration:
                return None

    rows = []
    calibration_methods = {
        "MinMax"     : CalibrationMethod.MinMax,
        "Entropy"    : CalibrationMethod.Entropy,
        "Percentile" : CalibrationMethod.Percentile,
    }

    for calib_name, calib_method in calibration_methods.items():
        print(f"        Calibration: {calib_name:12s}", end="", flush=True)
        int8_onnx = f"resnet50_int8_{calib_name.lower()}.onnx"
        try:
            quantize_static(
                model_input          = fp32_onnx,
                model_output         = int8_onnx,
                calibration_data_reader = CIFARCalibReader(calib_loader),
                quant_format         = QuantFormat.QDQ,
                per_channel          = True,
                reduce_range         = False,
                weight_type          = QuantType.QInt8,
                activation_type      = QuantType.QUInt8,
                calibrate_method     = calib_method,
                extra_options        = {
                    "EnableSubgraph"              : True,
                    "MatMulConstBOnly"            : False,
                    "AddQDQPairToWeight"          : True,
                    "QuantizeAllFixedZeroPointTensors": True,
                },
            )

            providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
                         if "CUDAExecutionProvider" in onnxruntime.get_available_providers()
                         else ["CPUExecutionProvider"])
            sess_opts = onnxruntime.SessionOptions()
            sess_opts.graph_optimization_level = (
                onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
            )
            session = onnxruntime.InferenceSession(int8_onnx, sess_opts,
                                                    providers=providers)
            active_provider = session.get_providers()[0]

            metrics  = evaluate_onnx(session, test_loader)
            disk_mb  = disk_size_mb(int8_onnx)

            # ORT latency (wall-clock)
            dummy_np   = np.random.randn(BATCH_SIZE, 3, 32, 32).astype(np.float32)
            input_name = session.get_inputs()[0].name
            for _ in range(5):
                session.run(None, {input_name: dummy_np})
            t0 = time.perf_counter()
            for _ in range(INFERENCE_RUNS):
                session.run(None, {input_name: dummy_np})
            mean_ms = (time.perf_counter() - t0) / INFERENCE_RUNS * 1000

            latency = {
                "mean_ms"               : round(mean_ms, 4),
                "batch_size"            : BATCH_SIZE,
                "throughput_imgs_per_sec": round(BATCH_SIZE / (mean_ms / 1000), 1),
                "timing_method"         : "wall-clock (time.perf_counter)",
            }

            # Standard 8-metric block — ONNX model has no .parameters(),
            # so we pass params from flops_fp32 (same architecture, same count).
            std_metrics = build_metrics_block(
                metrics           = metrics,
                baseline_acc      = baseline_acc,
                model_or_path     = int8_onnx,
                inference_latency = latency,
                flops_fp32        = flops_fp32,
            )

            row = {
                "backend"            : "onnxruntime_int8",
                "calibration_method" : calib_name,
                "execution_provider" : active_provider,
                "description"        : (
                    f"ONNX Runtime static INT8 PTQ (calibration={calib_name}). "
                    "QDQ format: QuantizeLinear/DequantizeLinear pairs wrap each op. "
                    "S = (x_max − x_min) / (2^8 − 1),  Z = round(−x_min / S). "
                    f"Calibrated on {CALIB_SIZE} CIFAR-10 training images. "
                    "GPU inference via CUDAExecutionProvider."
                ),
                "quantization_math"  : {
                    "S"       : "S = (x_max − x_min) / (2^8 − 1)   [255 levels for UINT8 activations]",
                    "Z"       : "Z = round(−x_min / S)",
                    "Q(x)"    : "Q(x) = clip(round(x / S) + Z, 0, 255)  → stored as UINT8",
                    "x̂"       : "x̂ = S · (Q(x) − Z)  → reconstructed at op boundary",
                    "ε_bound" : "|ε| ≤ S/2  (smaller S ↔ finer quantisation grid ↔ less error)",
                    "format"  : "QDQ (QuantizeLinear / DequantizeLinear node pairs, ONNX opset 10+)",
                },
                "ops_quantized"      : "Conv, MatMul, Gemm, Add (residual) — subject to graph partitioner",
                "weight_dtype"       : "INT8 per-channel symmetric",
                "activation_dtype"   : "UINT8 per-tensor affine",
                "calibration_samples": CALIB_SIZE,
                "disk_saved_mb"      : round(base_disk - std_metrics["size_disk_mb"], 4)
                                       if std_metrics["size_disk_mb"] else None,
                "compression_ratio"  : round(base_disk / std_metrics["size_disk_mb"], 4)
                                       if std_metrics["size_disk_mb"] else None,
                # ── All 8 evaluation metrics (flat, top-level) ──────────────
                **std_metrics,
            }
            print(f" → Acc: {metrics['accuracy']:.4f}  "
                  f"Drop: {baseline_acc - metrics['accuracy']:+.4f}  "
                  f"Disk: {std_metrics['size_disk_mb']:.2f} MB  "
                  f"Lat: {mean_ms:.1f} ms ({active_provider})")
            rows.append(row)

        except Exception as e:
            print(f" → FAILED: {e}")
            rows.append({
                "backend": "onnxruntime_int8",
                "calibration_method": calib_name,
                "description": f"FAILED: {e}",
                "accuracy": None,
            })

    for f in [fp32_onnx] + [f"resnet50_int8_{m.lower()}.onnx"
                              for m in calibration_methods]:
        if os.path.exists(f):
            os.remove(f)

    return rows


# ══════════════════════════════════════════════════════════════════════════════
# 2. TensorRT — INT8 PTQ with Calibration Cache
# ══════════════════════════════════════════════════════════════════════════════
def run_tensorrt_ptq(fp32_model, test_loader, calib_loader,
                     baseline_acc: float, base_disk: float,
                     device: str, flops_fp32: Dict) -> Dict:
    print("\n  [2/3] TensorRT — INT8 PTQ")
    try:
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit
    except ImportError as e:
        print(f"        ⚠ TensorRT/pycuda not available: {e}")
        return {
            "backend"    : "tensorrt_int8",
            "description": f"SKIPPED — TensorRT not installed: {e}",
            "accuracy"   : None,
            "install"    : "pip install tensorrt pycuda",
        }

    print("        Framework: TensorRT INT8 engine")
    print("        Fusion:    Conv-BN-ReLU automatic, memory layout optimized")
    print("        Calib:     IInt8EntropyCalibrator2 (KL-divergence minimization)")

    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
    fp32_onnx  = "resnet50_trt_fp32.onnx"

    try:
        export_to_onnx(fp32_model, fp32_onnx)

        class CIFARCalibrator(trt.IInt8EntropyCalibrator2):
            def __init__(self, loader, cache_file="trt_calib.cache"):
                super().__init__()
                self.loader     = iter(loader)
                self.batches    = 0
                self.cache_file = cache_file
                self.device_mem = None
                batch, _ = next(iter(loader))
                self.batch_size = batch.shape[0]
                self.buf_size   = int(np.prod(batch.shape) * 4)
                self.device_mem = cuda.mem_alloc(self.buf_size)

            def get_batch_size(self):
                return self.batch_size

            def get_batch(self, names):
                if self.batches >= CALIB_BATCHES:
                    return None
                try:
                    imgs, _ = next(self.loader)
                    cuda.memcpy_htod(self.device_mem, imgs.numpy().astype(np.float32))
                    self.batches += 1
                    return [int(self.device_mem)]
                except StopIteration:
                    return None

            def read_calibration_cache(self):
                if os.path.exists(self.cache_file):
                    with open(self.cache_file, "rb") as f:
                        return f.read()
                return None

            def write_calibration_cache(self, cache):
                with open(self.cache_file, "wb") as f:
                    f.write(cache)

        builder  = trt.Builder(TRT_LOGGER)
        network  = builder.create_network(
            1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
        )
        parser   = trt.OnnxParser(network, TRT_LOGGER)
        with open(fp32_onnx, "rb") as f:
            if not parser.parse(f.read()):
                raise RuntimeError(f"ONNX parse error: {parser.get_error(0)}")

        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
        config.set_flag(trt.BuilderFlag.INT8)
        config.set_flag(trt.BuilderFlag.FP16)

        calibrator = CIFARCalibrator(calib_loader)
        config.int8_calibrator = calibrator

        profile = builder.create_optimization_profile()
        profile.set_shape("input",
                           min=(1, 3, 32, 32),
                           opt=(BATCH_SIZE, 3, 32, 32),
                           max=(BATCH_SIZE * 2, 3, 32, 32))
        config.add_optimization_profile(profile)

        print("        Building TRT INT8 engine (this may take several minutes)...")
        t_build = time.perf_counter()
        serialized = builder.build_serialized_network(network, config)
        build_time = time.perf_counter() - t_build
        print(f"        Engine built in {build_time:.1f}s")

        trt_path = "resnet50_int8.trt"
        with open(trt_path, "wb") as f:
            f.write(serialized)

        runtime = trt.Runtime(TRT_LOGGER)
        engine  = runtime.deserialize_cuda_engine(serialized)
        context = engine.create_execution_context()
        context.set_input_shape("input", (BATCH_SIZE, 3, 32, 32))

        d_input  = cuda.mem_alloc(BATCH_SIZE * 3 * 32 * 32 * 4)
        d_output = cuda.mem_alloc(BATCH_SIZE * NUM_CLASSES * 4)
        stream   = cuda.Stream()

        def infer_trt(x_np):
            cuda.memcpy_htod_async(d_input, x_np.astype(np.float32), stream)
            context.execute_async_v2(
                bindings=[int(d_input), int(d_output)], stream_handle=stream.handle
            )
            out = np.empty((BATCH_SIZE, NUM_CLASSES), dtype=np.float32)
            cuda.memcpy_dtoh_async(out, d_output, stream)
            stream.synchronize()
            return out

        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
        preds, labels = [], []
        for inputs, lbls in test_loader:
            out = infer_trt(inputs.numpy())
            preds.extend(np.argmax(out, 1))
            labels.extend(lbls.numpy())
        y_pred, y_true = np.array(preds), np.array(labels)
        metrics = {
            "accuracy" : float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
            "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
            "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        }

        # GPU latency with CUDA events
        dummy_np = np.random.randn(BATCH_SIZE, 3, 32, 32).astype(np.float32)
        start_ev = cuda.Event()
        end_ev   = cuda.Event()
        for _ in range(5):
            infer_trt(dummy_np)
        timings = []
        for _ in range(INFERENCE_RUNS):
            start_ev.record()
            infer_trt(dummy_np)
            end_ev.record()
            end_ev.synchronize()
            timings.append(start_ev.time_till(end_ev))
        arr = np.array(timings)

        disk_mb = disk_size_mb(trt_path)
        latency = {
            "mean_ms"               : round(float(arr.mean()), 4),
            "std_ms"                : round(float(arr.std()),  4),
            "min_ms"                : round(float(arr.min()),  4),
            "max_ms"                : round(float(arr.max()),  4),
            "samples"               : INFERENCE_RUNS,
            "batch_size"            : BATCH_SIZE,
            "throughput_imgs_per_sec": round(BATCH_SIZE / (float(arr.mean()) / 1000), 1),
            "timing_method"         : "CUDA events (GPU-synchronized)",
        }

        std_metrics = build_metrics_block(
            metrics           = metrics,
            baseline_acc      = baseline_acc,
            model_or_path     = trt_path,
            inference_latency = latency,
            flops_fp32        = flops_fp32,
        )

        row = {
            "backend"           : "tensorrt_int8",
            "description"       : (
                "TensorRT INT8 engine: ONNX → TRT with IInt8EntropyCalibrator2. "
                "Layer fusion (Conv+BN+ReLU), memory layout optimization (NHWC), "
                "kernel autotuning at build time. "
                f"Engine built in {build_time:.1f}s; NOT portable across GPU architectures."
            ),
            "quantization_math" : {
                "calibration_procedure": (
                    "Collect per-tensor activation histogram over calib batches. "
                    "For each candidate threshold T: quantize histogram to 128 bins, "
                    "compute KL(original_hist || quantized_hist). "
                    "Select T* = argmin KL. Set S = T* / 127."
                ),
                "S_formula"    : "S = T* / 127  where T* minimizes histogram KL divergence",
                "dtype_weights": "INT8 per-channel symmetric",
                "dtype_acts"   : "INT8 per-tensor (calibrated threshold)",
                "fusion"       : "Conv+BN+ReLU fused to single kernel; residual add absorbed",
            },
            "build_time_sec"    : round(build_time, 2),
            "calibration_method": "IInt8EntropyCalibrator2 (histogram KL threshold search)",
            "calibration_samples": CALIB_SIZE,
            "disk_saved_mb"     : round(base_disk - disk_mb, 4),
            "compression_ratio" : round(base_disk / disk_mb, 4),
            # ── All 8 evaluation metrics (flat, top-level) ─────────────────
            **std_metrics,
        }
        print(f"        → Acc: {metrics['accuracy']:.4f}  "
              f"Drop: {baseline_acc - metrics['accuracy']:+.4f}  "
              f"Disk: {disk_mb:.2f} MB  GPU: {float(arr.mean()):.1f} ms")

    except Exception as e:
        print(f"        → FAILED: {e}")
        row = {
            "backend"    : "tensorrt_int8",
            "description": f"FAILED: {e}",
            "accuracy"   : None,
        }
    finally:
        for f in [fp32_onnx, "trt_calib.cache"]:
            if os.path.exists(f):
                os.remove(f)

    return row


# ══════════════════════════════════════════════════════════════════════════════
# 3. PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8
# ══════════════════════════════════════════════════════════════════════════════
def _resolve_pt2e_imports():
    """
    Resolve PT2E symbols across PyTorch 2.3 / 2.4 / 2.5+ import paths.
    Returns (prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn).
    """
    errors = []

    prepare_pt2e = convert_pt2e = None
    for mod_path in [
        "torch.ao.quantization.quantize_pt2e",
        "torch.quantization.quantize_pt2e",
    ]:
        try:
            import importlib
            m = importlib.import_module(mod_path)
            prepare_pt2e = m.prepare_pt2e
            convert_pt2e = m.convert_pt2e
            break
        except (ImportError, AttributeError) as e:
            errors.append(f"  {mod_path}: {e}")

    if prepare_pt2e is None:
        raise ImportError(
            "Cannot find prepare_pt2e / convert_pt2e. Tried:\n" + "\n".join(errors) +
            "\nRequires PyTorch >= 2.3."
        )

    Quantizer = get_config = None
    for mod_path, cls, cfg in [
        (
            "torch.ao.quantization.quantizer.x86_inductor_quantizer",
            "X86InductorQuantizer",
            "get_default_x86_inductor_quantization_config",
        ),
    ]:
        try:
            import importlib
            m = importlib.import_module(mod_path)
            Quantizer  = getattr(m, cls)
            get_config = getattr(m, cfg)
            break
        except (ImportError, AttributeError) as e:
            errors.append(f"  {mod_path}.{cls}: {e}")

    if Quantizer is None:
        raise ImportError(
            "Cannot find X86InductorQuantizer. Tried:\n" + "\n".join(errors)
        )

    export_fn = None
    try:
        from torch.export import export as _torch_export
        def export_fn(model, example_args):
            ep = _torch_export(model, example_args)
            return ep.module()
    except (ImportError, AttributeError) as e:
        errors.append(f"  torch.export.export: {e}")

    if export_fn is None:
        try:
            from torch._export import capture_pre_autograd_graph as _cap
            def export_fn(model, example_args):
                return _cap(model, example_args)
        except (ImportError, AttributeError) as e:
            errors.append(f"  torch._export.capture_pre_autograd_graph: {e}")

    if export_fn is None:
        raise ImportError("Cannot find a graph export function. Tried:\n" + "\n".join(errors))

    return prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn


def run_torch_cuda_ptq(fp32_model, test_loader, calib_loader,
                        baseline_acc: float, base_disk: float,
                        device: str, flops_fp32: Dict) -> List[Dict]:
    print("\n  [3/3] PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8")
    rows = []
    import torch.nn as nn

    # ── 3a. Dynamic PTQ on CUDA ────────────────────────────────────────────────
    print("        3a. Dynamic PTQ (Linear → qint8) on CUDA")
    try:
        q_dyn = torch.quantization.quantize_dynamic(
            copy.deepcopy(fp32_model), {nn.Linear}, dtype=torch.qint8
        )
        q_dyn = q_dyn.to(device).eval()

        metrics   = evaluate_torch(q_dyn, test_loader, device)
        lat       = measure_gpu_throughput_ms(q_dyn, device)
        params    = count_parameters(q_dyn)
        disk_mb   = disk_size_mb(q_dyn.cpu())

        std_metrics = build_metrics_block(
            metrics           = metrics,
            baseline_acc      = baseline_acc,
            model_or_path     = None,   # we already computed disk_mb above
            inference_latency = lat,
            flops_fp32        = flops_fp32,
            params_override   = params,
        )
        # Inject the correct disk size (q_dyn is in-memory; we serialised it above)
        std_metrics["size_disk_mb"] = round(disk_mb, 4)

        rows.append({
            "backend"         : "torch_cuda_dynamic",
            "description"     : (
                "PyTorch dynamic PTQ on CUDA: Linear weights stored as qint8 in memory. "
                "On CUDA, weights are dequantized to FP32/FP16 BEFORE the cuBLAS GEMM — "
                "this is NOT true INT8 GEMM. Benefit: memory bandwidth (~2× for Linear). "
                "Conv2d stays FP32. No calibration data needed."
            ),
            "quantization_math": {
                "scope"    : "Linear weights only (Conv2d stays FP32)",
                "storage"  : "qint8 (~2× memory reduction vs FP32)",
                "compute"  : "FP16/FP32 after weight dequant — NOT INT8 GEMM on GPU",
                "S"        : "computed per-tensor from weight min/max at runtime",
            },
            "ops_quantized"    : "Linear weights only (Conv2d stays FP32)",
            "calibration_samples": 0,
            "disk_saved_mb"    : round(base_disk - disk_mb, 4),
            "compression_ratio": round(base_disk / disk_mb, 4) if disk_mb else None,
            # ── All 8 evaluation metrics (flat, top-level) ─────────────────
            **std_metrics,
        })
        print(f"           → Acc: {metrics['accuracy']:.4f}  "
              f"GPU: {lat['mean_ms']:.1f} ms  Disk: {disk_mb:.2f} MB")
    except Exception as e:
        print(f"           → FAILED: {e}")
        rows.append({"backend": "torch_cuda_dynamic",
                     "description": f"FAILED: {e}", "accuracy": None})

    # ── 3b. PT2E + X86InductorQuantizer → torch.compile ──────────────────────
    print("        3b. PT2E (torch.export + X86InductorQuantizer) → torch.compile")
    try:
        prepare_pt2e, convert_pt2e, Quantizer, get_config, export_fn = (
            _resolve_pt2e_imports()
        )
        print(f"           PT2E imports resolved (torch {torch.__version__})")

        example_args = (torch.randn(1, 3, 32, 32, device=device),)
        m = copy.deepcopy(fp32_model).to(device).eval()
        exported = export_fn(m, example_args)

        quantizer = Quantizer()
        quantizer.set_global(get_config())
        prepared = prepare_pt2e(exported, quantizer)

        prepared.eval()
        with torch.no_grad():
            for i, (imgs, _) in enumerate(calib_loader):
                prepared(imgs.to(device))
                if i + 1 >= CALIB_BATCHES:
                    break

        q_pt2e     = convert_pt2e(prepared)
        q_compiled = torch.compile(q_pt2e, mode="max-autotune", backend="inductor")

        metrics = evaluate_torch(q_compiled, test_loader, device)
        lat     = measure_gpu_throughput_ms(q_compiled, device)
        params  = count_parameters(q_pt2e)   # compiled graph; use pre-compile for params

        std_metrics = build_metrics_block(
            metrics           = metrics,
            baseline_acc      = baseline_acc,
            model_or_path     = None,   # compiled model can't be serialised trivially
            inference_latency = lat,
            flops_fp32        = flops_fp32,
            params_override   = params,
        )
        # PT2E compiled model doesn't serialise to a simple .pth — mark N/A
        std_metrics["size_disk_mb"] = None

        rows.append({
            "backend"         : "torch_pt2e_inductor_int8",
            "torch_version"   : torch.__version__,
            "description"     : (
                "PT2E (PyTorch 2 Export) + X86InductorQuantizer + torch.compile. "
                "torch.export → prepare_pt2e (MinMaxObservers) → calibrate → "
                "convert_pt2e (freeze S/Z) → torch.compile(inductor). "
                "MATURITY WARNING (torch 2.3–2.5): some ops may fall back to FP16."
            ),
            "quantization_math": {
                "S"         : "S = (x_max − x_min) / (2^8 − 1)  [from MinMaxObserver]",
                "Z"         : "Z = round(−x_min / S)",
                "Q(x)"      : "Q(x) = clip(round(x / S) + Z, 0, 255)  → UINT8",
                "compute"   : "Inductor selects INT8 Triton or cuBLAS per op at compile time",
                "dtype_w"   : "INT8 symmetric per-channel",
                "dtype_act" : "UINT8 per-tensor affine",
            },
            "ops_quantized"   : "Conv2d + Linear (all ATen graph ops annotated by quantizer)",
            "calibration_samples": CALIB_SIZE,
            "disk_saved_mb"   : None,   # compiled model
            "compression_ratio": None,
            # ── All 8 evaluation metrics (flat, top-level) ─────────────────
            **std_metrics,
        })
        print(f"           → Acc: {metrics['accuracy']:.4f}  "
              f"GPU: {lat['mean_ms']:.1f} ms")

    except ImportError as e:
        msg = str(e)
        print(f"           → SKIPPED (PT2E unavailable): {msg}")
        rows.append({
            "backend"    : "torch_pt2e_inductor_int8",
            "description": f"SKIPPED — PT2E not available: {msg}",
            "accuracy"   : None,
            "install_hint": "pip install torch>=2.3",
        })
    except Exception as e:
        print(f"           → FAILED: {e}")
        rows.append({"backend": "torch_pt2e_inductor_int8",
                     "description": f"FAILED: {e}", "accuracy": None})

    return rows


# ══════════════════════════════════════════════════════════════════════════════
# FP32 GPU Baseline
# ══════════════════════════════════════════════════════════════════════════════
def benchmark_fp32_gpu(fp32_model, test_loader, device, flops_fp32: Dict) -> Dict:
    """Measure FP32 GPU baseline — includes all 8 evaluation metrics."""
    print(f"\n  [0/3] FP32 GPU Baseline ({device})")
    model   = copy.deepcopy(fp32_model).to(device).eval()
    metrics = evaluate_torch(model, test_loader, device)
    lat     = measure_gpu_throughput_ms(model, device)
    params  = count_parameters(model)
    disk_mb = disk_size_mb(model.cpu())
    ram_mb  = params["params_total"] * 4 / 1024 ** 2

    std_metrics = build_metrics_block(
        metrics           = metrics,
        baseline_acc      = metrics["accuracy"],  # drop vs self = 0
        model_or_path     = None,
        inference_latency = lat,
        flops_fp32        = flops_fp32,
        params_override   = params,
    )
    std_metrics["size_disk_mb"]  = round(disk_mb, 4)
    std_metrics["accuracy_drop"] = 0.0   # baseline has zero drop by definition

    print(f"        Acc: {metrics['accuracy']:.4f}  "
          f"GPU: {lat['mean_ms']:.1f} ms  Disk: {disk_mb:.2f} MB")
    return {
        "device"          : str(device),
        "gpu_name"        : torch.cuda.get_device_name(device) if torch.cuda.is_available() else "N/A",
        "ram_fp32_mb"     : round(ram_mb, 4),
        # ── All 8 evaluation metrics (flat, top-level) ─────────────────────
        **std_metrics,
    }


# ══════════════════════════════════════════════════════════════════════════════
# Evaluation Summary Table
# ══════════════════════════════════════════════════════════════════════════════
def build_evaluation_summary(fp32_baseline: Dict, results: List[Dict]) -> List[Dict]:
    """
    Produce a flat comparison table with all 8 metrics for every backend.
    Rows are sorted by accuracy_drop ascending (least degradation first).
    This table is the primary artefact for study and cross-backend comparison.
    """
    METRIC_KEYS = [
        "accuracy", "precision", "recall", "f1",
        "params_total", "params_M",
        "size_disk_mb",
        "inference_latency",   # keep full sub-dict for detail
        "flops_total", "gflops",
        "accuracy_drop",
    ]

    def _row(d: Dict, label: str) -> Dict:
        lat = d.get("inference_latency", {})
        return {
            "backend"                : label,
            "calibration_method"     : d.get("calibration_method", "—"),
            # Classification metrics
            "accuracy"               : d.get("accuracy"),
            "precision"              : d.get("precision"),
            "recall"                 : d.get("recall"),
            "f1"                     : d.get("f1"),
            "accuracy_drop"          : d.get("accuracy_drop"),
            # Model size & parameters
            "params_total"           : d.get("params_total"),
            "params_M"               : d.get("params_M"),
            "size_disk_mb"           : d.get("size_disk_mb"),
            # Inference time (flattened for easy tabulation)
            "latency_mean_ms"        : lat.get("mean_ms"),
            "latency_std_ms"         : lat.get("std_ms"),
            "throughput_imgs_per_sec": lat.get("throughput_imgs_per_sec"),
            # Operations
            "flops_total"            : d.get("flops_total"),
            "gflops"                 : d.get("gflops"),
        }

    rows = [_row(fp32_baseline, "fp32_gpu_baseline")]
    for r in results:
        if r.get("accuracy") is not None:
            rows.append(_row(r, r.get("backend", "unknown")))

    # Sort: baseline first, then by ascending accuracy_drop
    rows.sort(key=lambda r: (0 if r["backend"] == "fp32_gpu_baseline" else 1,
                              r["accuracy_drop"] if r["accuracy_drop"] is not None else 99))
    return rows


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════
def main():
    parser = argparse.ArgumentParser(description="GPU PTQ for ResNet-50/CIFAR-10")
    parser.add_argument("--install-deps", action="store_true",
                        help="Install required GPU packages and exit")
    parser.add_argument("--device", default="cuda",
                        help="Target device: cuda, cuda:0, cuda:1, cpu")
    args, _ = parser.parse_known_args()

    if args.install_deps:
        install_deps()
        return

    # ── Dependency check ───────────────────────────────────────────────────────
    missing = []
    if torch is None:
        missing.append("torch  →  pip install torch --index-url https://download.pytorch.org/whl/cu121")
    if onnx is None:
        missing.append("onnx   →  pip install onnx")
    if ort is None:
        missing.append("onnxruntime-gpu  →  pip install onnxruntime-gpu")
    if missing:
        print("\n⚠  Missing required packages:")
        for m in missing:
            print(f"   {m}")
        print("\nRun:  python gpu_ptq.py --install-deps")
        sys.exit(1)

    device = torch.device(args.device if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*65}")
    print("  GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10")
    print(f"  Device    : {device}")
    if torch.cuda.is_available():
        print(f"  GPU       : {torch.cuda.get_device_name(device)}")
        print(f"  CUDA      : {torch.version.cuda}")
        print(f"  cuDNN     : {torch.backends.cudnn.version()}")
    print("  Frameworks: ONNX Runtime GPU  |  TensorRT  |  PyTorch PT2E")
    print(f"{'='*65}")

    # ── Load baseline ──────────────────────────────────────────────────────────
    if not os.path.exists(BASELINE_MODEL_PATH):
        print(f"\n✗ Baseline model not found: {BASELINE_MODEL_PATH}")
        print("  Run the baseline training script first.")
        sys.exit(1)

    with open(BASELINE_METRICS_PATH) as f:
        baseline_metrics = json.load(f)
    baseline_acc = baseline_metrics["accuracy"]

    fp32_model   = load_baseline(BASELINE_MODEL_PATH)
    test_loader  = get_test_loader()
    calib_loader = get_calib_loader()

    # ── FLOPs (identical for INT8 and FP32; computed once on FP32 model) ───────
    print("\n  Computing FLOPs (op count — identical for FP32 and INT8)...")
    flops_fp32 = {}

    try:
        flops_fp32 = count_flops_pytorch_profiler(fp32_model.cpu(), device="cpu")
        gf = flops_fp32.get("gflops_fp32", 0)
        pm = flops_fp32.get("params_M", 0)
        print(f"  FLOPs (torch.profiler): {gf:.3f} GFLOPs | {pm:.1f}M params")
    except Exception as e:
        print(f"  torch.profiler failed: {e}")
        if fvcore is not None:
            try:
                flops_fp32 = count_flops_fvcore(fp32_model.cpu())
                print(f"  FLOPs (fvcore): {flops_fp32['gflops_fp32']:.3f} GFLOPs | "
                      f"{flops_fp32['params_M']:.1f}M params")
            except Exception as e2:
                print(f"  fvcore also failed: {e2}")
                flops_fp32 = count_flops_manual(fp32_model.cpu())
        else:
            flops_fp32 = count_flops_manual(fp32_model.cpu())
        gf = flops_fp32.get("gflops_fp32") or flops_fp32.get("gflops", 0)
        print(f"  FLOPs (manual): {gf:.3f} GFLOPs")

    # If params not yet in flops_fp32, add them now
    if "params_total" not in flops_fp32:
        p = count_parameters(fp32_model)
        flops_fp32["params_total"] = p["params_total"]
        flops_fp32["params_M"]     = p["params_M"]

    print("  Note: FLOPs are unchanged by quantization. "
          "INT8 speedup = hardware throughput (wider SIMD), not fewer ops.")

    base_disk = disk_size_mb(fp32_model)

    # ── FP32 GPU Baseline ──────────────────────────────────────────────────────
    fp32_gpu_baseline = benchmark_fp32_gpu(fp32_model, test_loader, device, flops_fp32)

    # ── Run PTQ backends ───────────────────────────────────────────────────────
    all_results = []
    all_results.extend(
        run_onnx_ptq(fp32_model, test_loader, calib_loader,
                     baseline_acc, base_disk, device, flops_fp32)
    )
    all_results.append(
        run_tensorrt_ptq(fp32_model, test_loader, calib_loader,
                         baseline_acc, base_disk, device, flops_fp32)
    )
    all_results.extend(
        run_torch_cuda_ptq(fp32_model, test_loader, calib_loader,
                           baseline_acc, base_disk, device, flops_fp32)
    )

    # ── Best config ────────────────────────────────────────────────────────────
    valid = [r for r in all_results if r.get("accuracy") is not None]
    best  = min(valid, key=lambda r: (r.get("accuracy_drop", 99),
                                       r.get("size_disk_mb") or 99)) if valid else {}

    # ── Evaluation summary table (all 8 metrics, all backends) ────────────────
    eval_summary = build_evaluation_summary(fp32_gpu_baseline, all_results)

    # ── Build report ───────────────────────────────────────────────────────────
    report = {
        "experiment"  : "GPU Post-Training Quantization (PTQ)",
        "description" : (
            "GPU-native PTQ using three frameworks: "
            "(1) ONNX Runtime GPU with QDQ format and 3 calibration methods, "
            "(2) TensorRT INT8 engine with entropy histogram calibration and layer fusion, "
            "(3) PyTorch CUDA: dynamic PTQ + PT2E Inductor INT8."
        ),
        # ── PRIMARY COMPARISON TABLE ───────────────────────────────────────────
        # evaluation_summary: flat list of dicts, one per backend, with all
        # 8 study metrics at top level. Sorted by accuracy_drop ascending.
        # Columns: backend, calibration_method, accuracy, precision, recall, f1,
        #          accuracy_drop, params_total, params_M, size_disk_mb,
        #          latency_mean_ms, latency_std_ms, throughput_imgs_per_sec,
        #          flops_total, gflops
        "evaluation_summary": eval_summary,
        # ── METRIC DEFINITIONS ─────────────────────────────────────────────────
        "metric_definitions": {
            "accuracy"               : "Top-1 classification accuracy on CIFAR-10 test set (10,000 images)",
            "precision"              : "Macro-averaged precision across all 10 classes (sklearn)",
            "recall"                 : "Macro-averaged recall across all 10 classes (sklearn)",
            "f1"                     : "Macro-averaged F1-score across all 10 classes (sklearn)",
            "accuracy_drop"          : "baseline_accuracy − quantized_accuracy (lower is better)",
            "params_total"           : "Total parameter count (integer); same architecture → same count across backends",
            "params_M"               : "params_total / 1e6  (millions)",
            "size_disk_mb"           : "Serialised model file size on disk in megabytes",
            "latency_mean_ms"        : "Mean batch inference time in milliseconds (GPU: CUDA events; ORT: wall-clock)",
            "latency_std_ms"         : "Standard deviation of batch inference time in milliseconds",
            "throughput_imgs_per_sec": "BATCH_SIZE / (latency_mean_ms / 1000) — images processed per second",
            "flops_total"            : (
                "Total floating-point operations = 2 × MACs. "
                "IDENTICAL for INT8 and FP32 — quantization does NOT change op count."
            ),
            "gflops"                 : "flops_total / 1e9",
        },
        "ptq_math": {
            "quantize"   : "Q(x) = clip(round(x / S) + Z, q_min, q_max)",
            "dequantize" : "x̂ = S · (Q(x) − Z)",
            "error"      : "ε = x − x̂,  |ε| ≤ S/2",
            "scale"      : "S = (x_max − x_min) / (2^b − 1)  [b=8 → 255 levels for UINT8]",
            "zero_point" : "Z = clip(round(−x_min / S), q_min, q_max)",
            "gpu_advantage_clarification": {
                "correct_framing"                   : (
                    "INT8 does NOT reduce FLOPs. The GPU speedup is a hardware property: "
                    "INT8 Tensor Core lanes process 4× more elements per cycle than FP32."
                ),
                "int8_tensor_core_throughput_ampere": "~4× vs FP32 (A100, RTX 30xx)",
                "int8_tensor_core_throughput_hopper": "~8× vs FP32 (H100)",
                "memory_bandwidth_reduction"        : "~4× (1 byte INT8 vs 4 bytes FP32 per weight)",
            },
        },
        "flops_methodology": {
            "tool"          : "torch.profiler (primary), fvcore, or manual hooks (fallback)",
            "formula_conv"  : "FLOPs = 2 × (Cin/groups) × Cout × Kh × Kw × Hout × Wout",
            "formula_linear": "FLOPs = 2 × in_features × out_features",
            "macs_to_flops" : "FLOPs = 2 × MACs",
            "critical_note" : (
                "FLOPs are IDENTICAL for INT8 and FP32 models. "
                "Use measured wall-clock latency or throughput (imgs/sec) for actual speedup."
            ),
        },
        "gpu_info": {
            "device"    : str(device),
            "gpu_name"  : fp32_gpu_baseline.get("gpu_name", "N/A"),
            "cuda_ver"  : getattr(torch.version, "cuda", "N/A") if torch else "N/A",
            "cudnn_ver" : str(torch.backends.cudnn.version()) if torch and torch.cuda.is_available() else "N/A",
        },
        "baseline_fp32"     : baseline_metrics,
        "baseline_fp32_gpu" : fp32_gpu_baseline,
        "calibration_config": {
            "calib_size"   : CALIB_SIZE,
            "calib_batches": CALIB_BATCHES,
            "dataset"      : "CIFAR-10 training split (first 1024 images)",
        },
        "best_config": {
            "backend"               : best.get("backend"),
            "calibration_method"    : best.get("calibration_method", best.get("observer")),
            # All 8 metrics for the best config
            "accuracy"              : best.get("accuracy"),
            "precision"             : best.get("precision"),
            "recall"                : best.get("recall"),
            "f1"                    : best.get("f1"),
            "accuracy_drop"         : best.get("accuracy_drop"),
            "params_total"          : best.get("params_total"),
            "params_M"              : best.get("params_M"),
            "size_disk_mb"          : best.get("size_disk_mb"),
            "latency_mean_ms"       : (best.get("inference_latency") or {}).get("mean_ms"),
            "throughput_imgs_per_sec": (best.get("inference_latency") or {}).get("throughput_imgs_per_sec"),
            "flops_total"           : best.get("flops_total"),
            "gflops"                : best.get("gflops"),
            "compression_ratio"     : best.get("compression_ratio"),
        } if best else {},
        "results"      : all_results,
        "framework_notes": {
            "onnxruntime_gpu"  : (
                "Requires onnxruntime-gpu. CUDAExecutionProvider dispatches to cuDNN INT8 "
                "(QLinearConv), cuBLASLt INT8 (QLinearMatMul). QDQ format."
            ),
            "tensorrt"         : (
                "Highest throughput for production GPU serving. Kernel autotuning at build time "
                "(GPU-specific binary — NOT portable). IInt8EntropyCalibrator2."
            ),
            "torch_pt2e"       : (
                "PT2E: torch.export → quantizer annotations → prepare_pt2e → calibrate → "
                "convert_pt2e → torch.compile(inductor). Still maturing on CUDA (torch 2.3–2.5)."
            ),
            "torch_dynamic_gpu": (
                "Stores Linear weights as INT8 but dequantizes to FP16 before cuBLAS GEMM. "
                "Saves memory bandwidth, NOT compute."
            ),
            "flops_vs_throughput": (
                "FLOPs (op count) are IDENTICAL for INT8 and FP32. "
                "The INT8 speedup is entirely a hardware throughput property."
            ),
        },
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    # ── Console summary ────────────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f"  ✓ Saved → {OUTPUT_JSON}")
    print(f"\n  {'Backend':<30} {'Acc':>7} {'Drop':>7} {'F1':>7} "
          f"{'Params(M)':>10} {'Disk(MB)':>9} {'Lat(ms)':>8} {'GFLOPs':>8}")
    print(f"  {'-'*90}")
    for row in eval_summary:
        lat_ms   = row.get("latency_mean_ms") or 0
        disk_mb  = row.get("size_disk_mb")
        disk_str = f"{disk_mb:.1f}" if disk_mb is not None else "  N/A"
        gf       = row.get("gflops") or 0
        print(f"  {row['backend']:<30} "
              f"{(row.get('accuracy') or 0):.4f}  "
              f"{(row.get('accuracy_drop') or 0):+.4f}  "
              f"{(row.get('f1') or 0):.4f}  "
              f"{(row.get('params_M') or 0):>10.2f}  "
              f"{disk_str:>9}  "
              f"{lat_ms:>8.1f}  "
              f"{gf:>8.3f}")

    if best:
        fp32_lat = (fp32_gpu_baseline.get("inference_latency") or {}).get("mean_ms", 1)
        best_lat = (best.get("inference_latency") or {}).get("mean_ms", 1)
        speedup  = fp32_lat / best_lat if best_lat else 0
        print(f"\n  Best    : {best.get('backend')}  "
              f"acc={best.get('accuracy', 'N/A'):.4f}  "
              f"drop={best.get('accuracy_drop', 'N/A'):+.4f}")
        print(f"  Speedup : {speedup:.2f}× vs FP32 GPU  "
              f"({best_lat:.1f} ms vs {fp32_lat:.1f} ms)")
    print(f"{'='*65}\n")


if __name__ == "__main__":
    main()


  GPU Post-Training Quantization (PTQ) — ResNet-50 / CIFAR-10
  Device    : cuda
  GPU       : NVIDIA GeForce RTX 5070 Laptop GPU
  CUDA      : 12.8
  cuDNN     : 92000
  Frameworks: ONNX Runtime GPU  |  TensorRT  |  PyTorch PT2E

  Computing FLOPs (op count — identical for FP32 and INT8)...
  torch.profiler failed: name 'count_flops_pytorch_profiler' is not defined
  FLOPs (manual): 2.596 GFLOPs
  Note: FLOPs are unchanged by quantization. INT8 speedup = hardware throughput (wider SIMD), not fewer ops.

  [0/3] FP32 GPU Baseline (cuda)
        Acc: 0.9320  GPU: 142.8 ms  Disk: 90.05 MB

  [1/3] ONNX Runtime — Static INT8 PTQ
        Framework: onnxruntime-gpu (CUDAExecutionProvider)
        Kernels:   cuDNN INT8 (Conv), cuBLASLt (MatMul), custom (Add)
        S/Z:       calibrated once from calib data; frozen in graph


W0429 11:35:02.454000 4920 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...


W0429 11:35:07.279000 4920 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


...

 → Acc: 0.9319  Drop: +0.0001  Disk: 90.18 MB  Lat: 190.7 ms (CUDAExecutionProvider)
        Calibration: Entropy     

Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 122
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128


 → Acc: 0.9319  Drop: +0.0001  Disk: 90.18 MB  Lat: 63.7 ms (CUDAExecutionProvider)
        Calibration: Percentile  

Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 122
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)


 → Acc: 0.9316  Drop: +0.0004  Disk: 90.18 MB  Lat: 63.7 ms (CUDAExecutionProvider)

  [2/3] TensorRT — INT8 PTQ
        ⚠ TensorRT/pycuda not available: No module named 'tensorrt'

  [3/3] PyTorch CUDA — Dynamic PTQ + PT2E Inductor INT8
        3a. Dynamic PTQ (Linear → qint8) on CUDA
           → FAILED: Could not run 'quantized::linear_dynamic' with arguments from the 'CUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'quantized::linear_dynamic' is only available for these backends: [CPU, Meta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradXLA, AutogradMPS, AutogradXPU, AutogradHPU, AutogradLazy, AutogradMTIA, AutogradMAIA,